In [31]:
from collections.abc import Callable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

In [32]:
def _check_limits(limits: list[float | int]) -> None:

    if not isinstance(limits, Sequence):
        raise TypeError(f"Expected Sequence, received {type(limits)}")

    limits_len = len(limits)
    if limits_len != 2:
        raise ValueError(f"Expected 2, received {limits_len}")

    start, end = limits

    if not isinstance(start, (float, int)) or not isinstance(end, (float, int)):
        raise TypeError(f"Expected float or int, received {type(start)} and {type(end)}")

    if start >= end:
        raise ValueError(f"Expected start < end, received {start} >= {end}")


In [33]:
def progonka(A: np.ndarray, B: np.ndarray, C: np.ndarray, D: np.ndarray) -> np.ndarray:
    """Solves a tridiagonal system with the Thomas algorithm (progonka method).

    The i-th equation of the system is:
        A[i] * y[i-1] + B[i] * y[i] + C[i] * y[i+1] = D[i]

    Parameters
    ----------
    A : np.ndarray
        Sub-diagonal, length n (A[0] is not used).
    B : np.ndarray
        Main diagonal, length n.
    C : np.ndarray
        Super-diagonal, length n (C[-1] is not used).
    D : np.ndarray
        Right-hand side, length n.

    Returns
    -------
    np.ndarray
        Solution vector y of length n.
    """
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    C = np.asarray(C, dtype=np.float64)
    D = np.asarray(D, dtype=np.float64)

    n = len(D)
    if not (len(A) == len(B) == len(C) == n):
        raise ValueError("A, B, C and D must have the same length")

    # Forward sweep
    alpha = np.empty(n)  # modified super-diagonal
    beta = np.empty(n)   # modified right-hand side

    alpha[0] = C[0] / B[0]
    beta[0] = D[0] / B[0]

    for i in range(1, n):
        denom = B[i] - A[i] * alpha[i - 1]
        alpha[i] = C[i] / denom
        beta[i] = (D[i] - A[i] * beta[i - 1]) / denom

    # Back substitution
    y = np.empty(n)
    y[-1] = beta[-1]
    for i in range(n - 2, -1, -1):
        y[i] = beta[i] - alpha[i] * y[i + 1]

    return y

# Завдання 1.1. Граничні умови 1-го роду (Діріхле) (4)
$$ y^{\prime\prime} + x^2 \cdot y^{\prime} + y = \sin{x}, x \in \left[ 0, 1 \right], y\left( 0 \right) = 1, y\left( 1 \right) = 0 $$

In [34]:
def solve_simple_dirichle(
        P: Callable[[np.ndarray[tuple[int], np.dtype[np.float64]]], np.ndarray[tuple[int], np.dtype[np.float64]]],
        Q: Callable[[np.ndarray[tuple[int], np.dtype[np.float64]]], np.ndarray[tuple[int], np.dtype[np.float64]]],
        R: Callable[[np.ndarray[tuple[int], np.dtype[np.float64]]], np.ndarray[tuple[int], np.dtype[np.float64]]],
        f: Callable[[np.ndarray[tuple[int], np.dtype[np.float64]]], np.ndarray[tuple[int], np.dtype[np.float64]]],
        y_start: float,
        y_end: float,
        limits: list[float | int],
        n: int = 100
    ) -> np.ndarray:
    """Solves a second-order linear ODE with Dirichle boundary conditions.

    The solver numerically approximates the solution $y(x)$ for the equation:
    $$P(x) \\cdot y^{\\prime\\prime} + Q(x) \\cdot y^{\\prime} + R(x) \\cdot y = f(x)$$

    Subject to the Dirichle boundary conditions:
    $$y(x_{start}) = y_{start}, \\quad y(x_{end}) = y_{end}$$

    Parameters
    ----------
    P, Q, R : Callable[[float], float]
        Functions representing the coefficients of the differential equation.
    f : Callable[[float], float]
        The right-hand side function.
    y_start : float
        The boundary value at the start of the interval.
    y_end : float
        The boundary value at the end of the interval.
    limits : list[float | int]
        A two-element list specifying the integration interval `[start, end]`.
    n : int, optional
        The number of internal grid points. The spatial grid $x$ is divided into `n + 2` segments. Default is 100.

    Returns
    -------
    numpy.ndarray
        A 1D array of shape `(n + 2,)` containing the approximated values of $y$ evaluated at each point in the discrete grid $x$.
    """

    _check_limits(limits)

    x = np.linspace(*limits, num=n + 2)
    h = x[1] - x[0]
    x_inside = x[1:-1]

    A: np.ndarray[tuple[int], np.dtype[np.float64]] = P(x_inside)/(h**2) - Q(x_inside)/(2*h)
    B: np.ndarray[tuple[int], np.dtype[np.float64]] = (-2*P(x_inside))/(h**2) + R(x_inside)
    C: np.ndarray[tuple[int], np.dtype[np.float64]] = P(x_inside)/(h**2) + Q(x_inside)/(2*h)
    D: np.ndarray[tuple[int], np.dtype[np.float64]] = f(x_inside)

    D[0] -= (P(x[0])/(h**2) - Q(x[0])/(2*h)) * y_start
    D[-1] -= (P(x[-1])/(h**2) + Q(x[-1])/(2*h)) * y_end
    # D[0] -= A[0] * y_start
    # D[-1] -= C[-1] * y_end

    y_inside = progonka(
        A=A,
        B=B,
        C=C,
        D=D
    )

    return np.concatenate(([y_start], y_inside, [y_end]))

solve_simple_dirichle(
    P=lambda x: 1 + 0*x,
    Q=lambda x: x**2,
    R=lambda x: 1 + 0*x,
    f=lambda x: np.sin(x),
    y_start=1,
    y_end=0,
    limits=[0, 1],
    n=4
)

array([1.        , 0.80294964, 0.57941042, 0.3554318 , 0.1550969 ,
       0.        ])

In [35]:
def exact(x):
    return np.sin(np.pi * x)

def f(x):
    return (1 - np.pi**2) * np.sin(np.pi * x) + np.pi * x**2 * np.cos(np.pi * x)

n = 99
x = np.linspace(0, 1, n + 2)
y = solve_simple_dirichle(
    P=lambda x: 1 + 0*x, Q=lambda x: x**2, R=lambda x: 1 + 0*x,
    f=f, y_start=0, y_end=0, limits=[0, 1], n=n
)
print(np.max(np.abs(y - exact(x))))

prev = None
for n in [9, 19, 39, 79, 159]:          # h = 1/(n+1): 1/10, 1/20, 1/40, ...
    x = np.linspace(0, 1, n + 2)
    y = solve_simple_dirichle(P=lambda x: 1 + 0*x, Q=lambda x: x**2, R=lambda x: 1 + 0*x,
                               f=f, y_start=0, y_end=0, limits=[0, 1], n=n)
    err = np.max(np.abs(y - exact(x)))
    print(n, err, "" if prev is None else f"ratio={prev/err:.2f}")
    prev = err

9.513568628594093e-05
9 0.009553891639233258 
19 0.0023800923833439125 ratio=4.01
39 0.0005945003242313618 ratio=4.00
79 0.00014864645491496908 ratio=4.00
159 3.7159762461547174e-05 ratio=4.00


# Завдання 1.2. Граничні умови 2-го роду (Неймана) (4)
$$ y^{\prime\prime} + x^2 \cdot y^{\prime} + y = \sin{x}, x \in \left[ 0, 1 \right], y^{\prime}\left( 0 \right) = 1, y^{\prime}\left( 1 \right) = 0 $$

# Завдання 1.3. Граничні умови 3-го роду (Робіна) (4)
$$ y^{\prime\prime} + x^2 \cdot y^{\prime} + y = \sin{x}, x \in \left[ 0, 1 \right], y\left( 0 \right) + y^{\prime}\left( 0 \right) = 1, 2 \cdot y\left( 1 \right) + y^{\prime}\left( 1 \right) = 0 $$


# Завдання 2. Граничні задачі для диференціальних рівнянь другого порядку у частинних похідних в декартовій системі координат (4)
$$
\frac{\partial}{\partial{x}} \left( \frac{k_{x}\left( x, y \right) \partial{u}}{\partial{x}} \right)
+ \frac{\partial}{\partial{y}} \left( \frac{k_{y}\left( x, y \right) \partial{u}}{\partial{x}} \right)
- q\left( x, y \right) \cdot u
= f\left( x, y \right)
$$
$$ \left( x, y \right) \in \left[ A; B \right] \times \left[ C; D \right] $$

$$ \left[ A; B \right] \times \left[ C; D \right] = \left[ 0.8, 2.5 \right] \times \left[ 1, 2 \right] $$
$$ k_{x} = 1 + x^{2} + 0.2 \cdot y^2, k_{y} = 1 + 0.5 \cdot x \cdot y, q = - \left( x + y \right), f = \cos{ \left( x \cdot y \right)} $$
$$ \frac{\partial{u}}{\partial{y}} \left( 0.8, y \right) = 0, u \left( 2.5, y \right) = \sin{y} $$

# Завдання 3. Рівняння Пуассона, яке задане у полярній системі координат (4)
$$
\Delta{u \left( r, \varphi \right)}
= \frac{1}{r} \cdot \frac{\partial}{\partial{r}} \left( r \cdot \frac{\partial{u}}{\partial{r}} \right)
+ \frac{1}{r^2} \cdot \frac{\partial^2{u}}{\partial{\varphi^2}}
= f \left( r, \varphi \right)
$$

$$
r \in \left[ 0.7; 2.0 \right],
\varphi \in \left[ 0, \frac{\pi}{4} \right],
f \left( r, \varphi \right) = r \cdot \sin{\varphi}
$$

$$
u \left( 0.7, \varphi \right) = 0,
\frac{\partial{u}}{\partial{r}}\left( 2.0, \varphi \right) = \varphi^2
$$
$$
2 \cdot u + 3 \cdot \frac{\partial{u}}{\partial{\varphi}}\left( r, 0 \right) = 1, 
u - 2 \cdot \frac{\partial{u}}{\partial{\varphi}}\left( r, \frac{\pi}{4} \right) = 0 

$$


# Завдання 4. Рівняння Пуассона, яке задане у полярній системі координат (осесиметричний випадок) (4)
$$
\Delta u(r)
= \frac{1}{r} \frac{d}{dr} \left( r \frac{\partial u}{\partial r} \right)
= f(r)
$$

$$
\begin{gathered}
r \in [1, 3], \quad f(r) = r \\
u(1) + \frac{du}{dr}(1) = 0, \quad u(3) = \cos 3
\end{gathered}
$$


# Завдання 5. Рівняння Пуассона у декартових тривимірних координатах (4)
$$
\frac{\partial^2 u}{\partial x^2}
+ \frac{\partial^2 u}{\partial y^2}
+ \frac{\partial^2 u}{\partial z^2}
= f(x, y, z)
$$

$$
\begin{gathered}
x \in [0; 2], \, y \in [0, 2], \, z \in [0, 1], \\
f(x, y, z) = e^{-x-y-z} \\
u(0, y, z) + \frac{\partial u}{\partial x}(0, y, z) = 1, \quad u(2, y, z) = 1 \\
\frac{\partial u}{\partial y}(x, 0, z) = 0, \quad u(x, 2, z) - 2\frac{\partial u}{\partial y}(x, 2, z) = 1 \\
u(x, y, 0) = 0, \quad \frac{\partial u}{\partial z}(x, y, 1) = 2
\end{gathered}
$$